In [9]:
with open("organic_soils.in", 'r') as file:
    organic_soils = [line.strip() for line in file]

import json

legend = '/mnt/slurm/jobs/global_soc/legend_v20250403.json'


with open(legend, "r") as f:
    data = json.load(f)
    
available_organic_st = []
available_organic_idx = []
unavailable_organic_st = []
for organic_st in organic_soils:
    for key in data:
        st_name = data[key]
        if st_name == organic_st:
            available_organic_st += [organic_st]
            available_organic_idx += [int(key)]
            break
    if st_name != organic_st:
        unavailable_organic_st += [organic_st]


In [45]:
with open("organic_soil_list.txt", "w") as f:
    for line in available_organic_st:
        f.write(f"{line}\n")

In [21]:
with open("organic_soil.in", "w") as f:
    for line in available_organic_st:
        f.write(f"soil.types_ensemble.{line}_p_30m_s_20000101_20221231_g_epsg.4326_v20250403.tif\n")

In [12]:
from joblib import dump, load
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

# final_model_LGB = load('/mnt/wodan/global_soc/scikit-map/v20250321/models/LGB_soil.types_v20250403.lz4')
# final_model_RF = load('/mnt/wodan/global_soc/scikit-map/v20250321/models/RF_soil.types_v20250403.lz4')
y = load('/mnt/wodan/global_soc/scikit-map/y_train_soil.type_v20250403.lz4')
# X = load('/mnt/wodan/global_soc/scikit-map/X_train_soil.type_v20250403.lz4')
# y_prob_hat_RF = final_model_RF.predict_proba(X)
# y_prob_hat_LGB = final_model_LGB.predict(X, num_iteration=final_model_LGB.best_iteration)
# y_prob_hat = (y_prob_hat_RF + y_prob_hat_LGB) / 2

y_prob_hat = load('/mnt/wodan/global_soc/scikit-map/y_pred_soil.type_v20250403.lz4')

y_org = np.zeros(y.shape)
for org_class in available_organic_idx:
    y_org[y == org_class] = 1

y_prob_org_hat = np.zeros(y.shape)
for org_class in available_organic_idx:
    y_prob_org_hat = y_prob_org_hat + y_prob_hat[:,org_class]
    
best_score = 0
best_th_test = 0
for th_test in np.linspace(0, 1, 51):
    y_org_hat = (y_prob_org_hat > th_test).astype(int)
    score = f1_score(y_org, y_org_hat)
    if score > best_score:
        best_score = score
        best_th_test = th_test

In [36]:

best_score = 0
best_th_test = 0
for th_test in np.linspace(0, 1, 101):
    y_org_hat = (y_prob_org_hat > th_test).astype(int)
    score = f1_score(y_org, y_org_hat)
    if score > best_score:
        best_score = score
        best_th_test = th_test
    
y_org_hat_best = (y_prob_org_hat > best_th_test).astype(int)
accuracy = accuracy_score(y_org, y_org_hat_best)

In [44]:
y_org_hat_best = (y_prob_org_hat > best_th_test).astype(int)

accuracy = accuracy_score(y_org, y_org_hat_best)
accuracy

0.9836056649821722

In [38]:
best_score

0.5751234422760404

In [40]:
correct = y_org_hat_best == y_org

In [42]:
np.sum(correct.astype(int))/correct.size

0.9836056649821722

In [43]:
from sklearn.metrics import accuracy_score
